# Package

In [82]:
# ==========================================
# Chargement
# ==========================================
import os
import pickle
import numpy as np
import pandas as pd

from __future__ import annotations
from typing import Any, Callable, Dict, Iterable, List, Optional, Tuple, Union

from dateutil.relativedelta import relativedelta

import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Chargement des résultats

In [83]:
# ---------- Helpers ----------

def load_any(path):
    """Charge un objet avec joblib puis pickle si besoin."""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Fichier introuvable: {path}")
    try:
        return joblib.load(path)
    except Exception:
        with open(path, "rb") as f:
            return pickle.load(f)

def load_with_fallbacks(primary_path, *alt_paths, expect_type=None, label=None):
    """
    Charge primary_path puis essaie les alt_paths en fallback.
    - expect_type: type attendu (ex: dict), si fourni on vérifie isinstance.
    - label: nom lisible pour logs (ex: 'Ridge bundle OOS').
    """
    name = f" ({label})" if label else ""
    try:
        obj = load_any(primary_path)
        if expect_type and not isinstance(obj, expect_type):
            print(f"⚠️ Type inattendu{name} pour {primary_path}: {type(obj)} (attendu {expect_type}).")
        else:
            return obj
    except Exception as e:
        print(f"ℹ️ Impossible de charger {primary_path}{name} : {e}")

    for p in alt_paths:
        try:
            obj = load_any(p)
            print(f"ℹ️ Fallback utilisé{name} → {p}")
            if expect_type and not isinstance(obj, expect_type):
                print(f"⚠️ Type inattendu{name} pour {p}: {type(obj)} (attendu {expect_type}).")
            return obj
        except Exception as e:
            print(f"ℹ️ Fallback raté{name} → {p} : {e}")

    print(f"⚠️ Aucun fichier disponible{name} (essayé: {[primary_path, *alt_paths]})")
    return None

def try_read_meta(path):
    """Lit un CSV méta. Tente index_col=0 puis sans index si échec."""
    for use_index in (0, None):
        try:
            meta = pd.read_csv(path, index_col=use_index)
            # Si 1 colonne, exposer Series
            if isinstance(meta, pd.DataFrame) and meta.shape[1] == 1:
                meta = meta.iloc[:, 0]
            return meta
        except Exception as e:
            last_err = e
    print(f"⚠️ Impossible de lire {path} : {last_err}")
    return None

def safe_meta_print(meta, keys, title="Méta"):
    """Affiche les clés demandées sans lever d'erreur si elles manquent."""
    if meta is None:
        print("⚠️ Pas de méta disponible.")
        return
    if isinstance(meta, pd.Series):
        d = meta.to_dict()
    elif isinstance(meta, pd.DataFrame):
        d = meta.iloc[0].to_dict() if len(meta) else {}
    elif isinstance(meta, dict):
        d = meta
    else:
        print(f"(info) Type de méta inattendu: {type(meta)}")
        d = {}

    print(f"\n--- {title} (clé: valeur) ---")
    for k in keys:
        print(f"{k}: {d.get(k, None)}")

def _normalize_month_start(s):
    s = pd.to_datetime(s, errors="coerce")
    if isinstance(s, pd.Series):
        return s.dt.to_period("M").dt.to_timestamp(how="start")
    if isinstance(s, pd.DatetimeIndex):
        return s.to_period("M").to_timestamp(how="start")
    return pd.Timestamp(s).to_period("M").to_timestamp(how="start")

def ensure_ms_index_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.index = pd.to_datetime(out.index).to_period("M").to_timestamp(how="start")
    return out.sort_index().asfreq("MS")

def ensure_features_present(df: pd.DataFrame, features: list[str], target_col: str | None = None):
    missing = [c for c in (features or []) if c not in df.columns]
    if target_col is not None and target_col not in df.columns:
        missing.append(target_col)
    if missing:
        raise ValueError(f"Colonnes manquantes dans le DataFrame: {missing}")

def make_exp_from_loaded(*, models, features, train_periods, preprocs=None, step_months: int = 12) -> dict:
    if not isinstance(models, (list, tuple)) or len(models) == 0:
        raise ValueError("'models' doit être une liste non vue.")
    if not isinstance(features, (list, tuple)) or len(features) == 0:
        raise ValueError("'features' doit être une liste non vide.")
    if not isinstance(train_periods, (list, tuple)) or len(train_periods) != len(models):
        raise ValueError("'train_periods' doit avoir la même longueur que 'models'.")
    if preprocs is not None and len(preprocs) != len(models):
        raise ValueError("'preprocs' doit avoir la même longueur que 'models' si fourni.")
    return {
        "models":        list(models),
        "features":      list(features),
        "preprocs":      list(preprocs) if preprocs is not None else None,
        "train_periods": list(pd.to_datetime(pd.Index(train_periods))),
        "step_months":   int(step_months),
    }

def exp_from_linreg_bundle_no_refit(linreg_bundle: dict) -> dict:
    if not isinstance(linreg_bundle, dict):
        raise ValueError("linreg_bundle invalide (pas un dict).")
    params = linreg_bundle.get("params", {}) or {}
    features = params.get("features", linreg_bundle.get("features", None))
    if not features:
        raise ValueError("Features absentes dans le bundle (params.features).")
    models = linreg_bundle.get("models", None)
    if not isinstance(models, (list, tuple)) or len(models) == 0:
        raise RuntimeError(
            "Ce bundle ne contient pas la liste des modèles par fenêtre ('models'). "
            "Sauvegarde les modèles lors de l'apprentissage, puis recharge."
        )
    preprocs = linreg_bundle.get("preprocs", None)
    if preprocs is not None and len(preprocs) != len(models):
        raise RuntimeError("'preprocs' n'a pas la même longueur que 'models'.")
    train_periods = linreg_bundle.get("train_fit_dates", None)
    if train_periods is None or len(train_periods) != len(models):
        raise RuntimeError("'train_fit_dates' manquant ou de longueur différente de 'models'.")
    return {
        "models":        list(models),
        "features":      list(features),
        "preprocs":      list(preprocs) if preprocs is not None else None,
        "train_periods": list(pd.to_datetime(pd.Index(train_periods))),
        "step_months":   12,
    }

def extract_last_model_from_bundle(bundle: dict, model_key="models"):
    """Récupère le dernier modèle depuis un bundle, sinon None."""
    try:
        if isinstance(bundle, dict) and model_key in bundle and bundle[model_key]:
            return bundle[model_key][-1]
    except Exception as e:
        print(f"⚠️ Impossible d'extraire le dernier modèle: {e}")
    return None

In [84]:
from pathlib import Path

ARTIFACT_DIR = Path.cwd()   # ajuste si besoin

# ----------------------------
# AR(1) avec CI
# ----------------------------
AR1_BUNDLE     = ARTIFACT_DIR / "AR1_CI_h12_oos_bundle.pkl"
AR1_LAST_PKL   = ARTIFACT_DIR / "AR1_CI_last_trained_model.pkl"
AR1_LAST_META  = ARTIFACT_DIR / "AR1_CI_last_trained_model_meta.csv"

# (fallbacks si jamais tu as encore des anciens fichiers)
AR1_BUNDLE_OLD   = ARTIFACT_DIR / "AR1_h12_oos_bundle.pkl"
AR1_LAST_PKL_OLD = ARTIFACT_DIR / "AR1_last_trained_model.pkl"
AR1_LAST_META_OLD= ARTIFACT_DIR / "AR1_last_trained_model_meta.csv"

In [85]:
# -*- coding: utf-8 -*-
import os
import pandas as pd

# ==============================================================
# ✅ Helper CSV fallback (utilise try_read_meta existante)
# ==============================================================

def try_read_meta_with_fallbacks(primary_path, *alt_paths, label=None):
    name = f" ({label})" if label else ""
    for p in (primary_path, *alt_paths):
        if p is None:
            continue
        if not os.path.exists(str(p)):
            continue
        meta = try_read_meta(str(p))  # fonction existante
        if meta is not None:
            print(f"ℹ️ Meta CSV chargé{name} → {p}")
            return meta
    print(f"⚠️ Aucun CSV méta disponible{name}")
    return None

# ==============================================================
# ✅ Helpers bundle
# ==============================================================

def is_bundle(x):
    return isinstance(x, dict) and any(k in x for k in (
        "params", "models", "train_fit_dates",
        "oos", "oos_predictions", "forecasts"
    ))

def load_bundle_or_model(*paths, label=None):
    """
    Charge le premier fichier existant parmi paths via load_with_fallbacks.
    Retourne: (bundle, model)
    """
    obj = load_with_fallbacks(
        *[str(p) for p in paths if p is not None],
        expect_type=None,
        label=label
    )
    if obj is None:
        return None, None
    if is_bundle(obj):
        return obj, None
    return None, obj

# ==============================================================
# ✅ Chargement — AR(1) UNIQUEMENT (robuste NameError)
# ==============================================================

print("\n=== Chargement AR(1) ===")

AR1_BUNDLE_    = globals().get("AR1_BUNDLE")
AR1_LAST_PKL_  = globals().get("AR1_LAST_PKL")
AR1_PKL_       = globals().get("AR1_PKL")        # optionnel
AR1_LAST_META_ = globals().get("AR1_LAST_META")

ar1_bundle, ar1_model = load_bundle_or_model(
    AR1_BUNDLE_,
    AR1_LAST_PKL_,
    AR1_PKL_,
    label="AR1"
)

ar1_meta = try_read_meta_with_fallbacks(
    AR1_LAST_META_,
    label="AR1 meta"
)

# ==============================================================
# ✅ Récapitulatif minimal
# ==============================================================

print("\n=== Récap chargement AR(1) ===")
ok = (ar1_bundle is not None) or (ar1_model is not None)
print(f"AR1 : {'OK' if ok else '—'}")

if ar1_bundle is not None:
    print(" → bundle AR1 chargé")
elif ar1_model is not None:
    print(" → modèle AR1 chargé (hors bundle)")



=== Chargement AR(1) ===
ℹ️ Meta CSV chargé (AR1 meta) → d:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\3_notebook\AR1_CI_last_trained_model_meta.csv

=== Récap chargement AR(1) ===
AR1 : OK
 → bundle AR1 chargé


In [86]:
from pathlib import Path
import joblib


# --- chemins CI (priorité) + fallback anciens noms ---
AR1_LAST_PKL_CI   = Path(AR1_LAST_PKL)   # ex: "AR1_CI_last_trained_model.pkl"
AR1_LAST_META_CI  = Path(AR1_LAST_META)  # ex: "AR1_CI_last_trained_model_meta.csv"
AR1_BUNDLE_CI     = Path(AR1_BUNDLE)     # ex: "AR1_CI_h12_oos_bundle.pkl"

AR1_LAST_PKL_OLD  = Path("AR1_last_trained_model.pkl")
AR1_LAST_META_OLD = Path("AR1_last_trained_model_meta.csv")
AR1_BUNDLE_OLD    = Path("AR1_h12_oos_bundle.pkl")

print("Fichiers présents ?")
for name, p in {
    "AR1_LAST_PKL (CI)":  AR1_LAST_PKL_CI,
    "AR1_LAST_META (CI)": AR1_LAST_META_CI,
    "AR1_BUNDLE (CI)":    AR1_BUNDLE_CI,
    "AR1_LAST_PKL (OLD)": AR1_LAST_PKL_OLD,
    "AR1_LAST_META (OLD)":AR1_LAST_META_OLD,
    "AR1_BUNDLE (OLD)":   AR1_BUNDLE_OLD,
}.items():
    print(f"{name:<18} : {'OK' if p.exists() else 'ABSENT'}")

# -------- Chargement modèle (CI en priorité) --------
model_path = AR1_LAST_PKL_CI if AR1_LAST_PKL_CI.exists() else AR1_LAST_PKL_OLD
ar1 = load_any(str(model_path)) if model_path.exists() else None

def label(x):
    return x.__class__.__name__ if x is not None else None

print("\nLabel modèle :")
print("AR1 :", label(ar1))
print("Chargé depuis :", model_path if model_path.exists() else None)

Fichiers présents ?
AR1_LAST_PKL (CI)  : OK
AR1_LAST_META (CI) : OK
AR1_BUNDLE (CI)    : OK
AR1_LAST_PKL (OLD) : OK
AR1_LAST_META (OLD) : OK
AR1_BUNDLE (OLD)   : OK

Label modèle :
AR1 : AutoRegResultsWrapper
Chargé depuis : d:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\3_notebook\AR1_CI_last_trained_model.pkl


In [87]:
# Chargement du bundle AR(1) (CI en priorité)
ar1_bundle = load_with_fallbacks(
    str(AR1_BUNDLE), str(AR1_BUNDLE_OLD),
    expect_type=dict,
    label="AR1 bundle"
)

print("Keys:", ar1_bundle.keys() if ar1_bundle else None)

if ar1_bundle is not None:
    oos = ar1_bundle.get("oos_predictions")
    print("oos_predictions shape:", None if oos is None else oos.shape)
    print(oos.head() if oos is not None else "⚠️ oos_predictions manquant")

    # ✅ check des colonnes CI
    if oos is not None:
        cols = oos.columns.tolist()
        print("\nColonnes:", cols)

        has_ci = ("y_hat_lo_95" in cols and "y_hat_hi_95" in cols) or \
                 ("y_pred_lo_95" in cols and "y_pred_hi_95" in cols)

        print(f"\nCI présent ? {'✅ OUI' if has_ci else '❌ NON'}")


Keys: dict_keys(['oos_predictions', 'params', 'meta'])
oos_predictions shape: (8892, 7)
      cutoff       date  h  y_true    y_pred  y_pred_lo_95  y_pred_hi_95
0 1962-12-01 1963-01-01  1    -0.1 -0.446879           NaN           NaN
1 1962-12-01 1963-02-01  2     0.4 -0.398027           NaN           NaN
2 1962-12-01 1963-03-01  3     0.1 -0.353101           NaN           NaN
3 1962-12-01 1963-04-01  4     0.1 -0.311787           NaN           NaN
4 1962-12-01 1963-05-01  5     0.4 -0.273793           NaN           NaN

Colonnes: ['cutoff', 'date', 'h', 'y_true', 'y_pred', 'y_pred_lo_95', 'y_pred_hi_95']

CI présent ? ✅ OUI


In [88]:
oos.head()

,cutoff,date,h,y_true,y_pred,y_pred_lo_95,y_pred_hi_95
0,1962-12-01,1963-01-01,1,-0.1,-0.446879,NaN,NaN
1,1962-12-01,1963-02-01,2,0.4,-0.398027,NaN,NaN
2,1962-12-01,1963-03-01,3,0.1,-0.353101,NaN,NaN
3,1962-12-01,1963-04-01,4,0.1,-0.311787,NaN,NaN
4,1962-12-01,1963-05-01,5,0.4,-0.273793,NaN,NaN


# Etablir de dataframe des résultats.

In [89]:
import numpy as np
import pandas as pd

# ==============================================================
# Standardisation AR(1) avec intervalles (CI inclus)
# ==============================================================

def build_df_from_bundle(bundle, default_method="AR1"):
    """
    Transforme un bundle OOS en DataFrame standardisé :
    date | true | pred | lo | hi | method
    """
    if not isinstance(bundle, dict) or "oos_predictions" not in bundle:
        raise ValueError("Bundle invalide ou oos_predictions manquant.")

    oos = bundle["oos_predictions"]
    if not isinstance(oos, pd.DataFrame):
        oos = pd.DataFrame(oos)

    out = oos.copy()

    # ----- vérité -----
    if "true" not in out.columns:
        if "y_true" in out.columns:
            out["true"] = out["y_true"]
        else:
            raise KeyError("Colonne vérité absente (true / y_true).")

    # ----- prédiction -----
    if "pred" not in out.columns:
        if "y_pred" in out.columns:
            out["pred"] = out["y_pred"]
        elif "y_hat" in out.columns:
            out["pred"] = out["y_hat"]
        else:
            raise KeyError("Colonne prédiction absente (pred / y_pred / y_hat).")

    # ----- intervalles (fallbacks robustes) -----
    if "lo" not in out.columns:
        if "y_pred_lo_95" in out.columns:
            out["lo"] = out["y_pred_lo_95"]
        elif "y_hat_lo_95" in out.columns:
            out["lo"] = out["y_hat_lo_95"]
        else:
            out["lo"] = np.nan

    if "hi" not in out.columns:
        if "y_pred_hi_95" in out.columns:
            out["hi"] = out["y_pred_hi_95"]
        elif "y_hat_hi_95" in out.columns:
            out["hi"] = out["y_hat_hi_95"]
        else:
            out["hi"] = np.nan

    # ----- normalisation date (month start) -----
    out["date"] = (
        pd.to_datetime(out["date"], errors="coerce")
          .dt.to_period("M")
          .dt.to_timestamp(how="start")
    )

    out["method"] = default_method

    return out[["date", "true", "pred", "lo", "hi", "method"]]

In [90]:
df_b = build_df_from_bundle(ar1_bundle, default_method="AR1")
print(df_b.head())
print(df_b.tail())

        date  true      pred  lo  hi method
0 1963-01-01  -0.1 -0.446879 NaN NaN    AR1
1 1963-02-01   0.4 -0.398027 NaN NaN    AR1
2 1963-03-01   0.1 -0.353101 NaN NaN    AR1
3 1963-04-01   0.1 -0.311787 NaN NaN    AR1
4 1963-05-01   0.4 -0.273793 NaN NaN    AR1
           date  true      pred        lo        hi method
8887 2025-04-01   0.3  0.207640 -0.839379  0.892549    AR1
8888 2025-05-01   0.2  0.185746 -0.827589  0.854198    AR1
8889 2025-06-01   0.0  0.166078 -0.718921  0.816778    AR1
8890 2025-07-01   0.0  0.148409 -0.444550  0.837898    AR1
8891 2025-08-01   0.1  0.132535 -0.244764  0.808580    AR1


In [91]:
n_ci = df_b[["lo","hi"]].notna().all(axis=1).sum()
print("Nb lignes avec CI non-NaN :", n_ci)

print("Premières lignes avec CI :")
print(df_b[df_b["lo"].notna() & df_b["hi"].notna()].head(3))

Nb lignes avec CI non-NaN : 8460
Premières lignes avec CI :
          date  true      pred        lo        hi method
432 1966-01-01  -0.9 -0.926708 -1.441494 -0.564588    AR1
433 1966-02-01  -1.3 -0.859940 -1.439143 -0.484898    AR1
434 1966-03-01  -0.9 -0.799117 -1.502424 -0.399817    AR1


In [92]:
df_b

,date,true,pred,lo,hi,method
0,1963-01-01,-0.1,-0.446879,NaN,NaN,AR1
1,1963-02-01,0.4,-0.398027,NaN,NaN,AR1
2,1963-03-01,0.1,-0.353101,NaN,NaN,AR1
3,1963-04-01,0.1,-0.311787,NaN,NaN,AR1
4,1963-05-01,0.4,-0.273793,NaN,NaN,AR1
...,...,...,...,...,...,...
8887,2025-04-01,0.3,0.207640,-0.839379,0.892549,AR1
8888,2025-05-01,0.2,0.185746,-0.827589,0.854198,AR1
8889,2025-06-01,0.0,0.166078,-0.718921,0.816778,AR1
8890,2025-07-01,0.0,0.148409,-0.444550,0.837898,AR1


In [93]:
df_b["date"].duplicated().sum()

np.int64(8140)

In [94]:
df_b.loc[df_b["date"].duplicated(keep=False)].sort_values("date")

,date,true,pred,lo,hi,method
1,1963-02-01,0.4,-0.398027,NaN,NaN,AR1
12,1963-02-01,0.4,-0.068810,NaN,NaN,AR1
2,1963-03-01,0.1,-0.353101,NaN,NaN,AR1
24,1963-03-01,0.1,0.401057,NaN,NaN,AR1
13,1963-03-01,0.1,-0.040257,NaN,NaN,AR1
...,...,...,...,...,...,...
8867,2025-06-01,0.0,0.131343,-0.497194,0.807388,AR1
8878,2025-06-01,0.0,0.211226,-0.571631,0.900716,AR1
8889,2025-06-01,0.0,0.166078,-0.718921,0.816778,AR1
8879,2025-07-01,0.0,0.189149,-0.337558,0.865194,AR1


In [95]:
from pathlib import Path
import pickle
import joblib

ARTIFACT_DIR = Path.cwd()

AR1_BUNDLE_CI = ARTIFACT_DIR / "AR1_CI_h12_oos_bundle.pkl"
AR1_MODEL_CI  = ARTIFACT_DIR / "AR1_CI_last_trained_model.pkl"
AR1_META_CI   = ARTIFACT_DIR / "AR1_CI_last_trained_model_meta.csv"

# ---------- Bundle ----------
if not AR1_BUNDLE_CI.exists():
    raise FileNotFoundError("Bundle AR1 CI introuvable")

with open(AR1_BUNDLE_CI, "rb") as f:
    ar1_bundle = pickle.load(f)

print("✅ Bundle AR1 CI chargé")
print("Keys:", ar1_bundle.keys())

# ---------- Modèle (optionnel) ----------
ar1_model = None
if AR1_MODEL_CI.exists():
    try:
        ar1_model = joblib.load(AR1_MODEL_CI)
        print("✅ Modèle AR1 CI chargé")
    except Exception:
        with open(AR1_MODEL_CI, "rb") as f:
            ar1_model = pickle.load(f)
        print("✅ Modèle AR1 CI chargé (pickle)")
else:
    print("⚠️ Modèle AR1 CI absent (normal si on fait seulement de l’évaluation)")


✅ Bundle AR1 CI chargé
Keys: dict_keys(['oos_predictions', 'params', 'meta'])
✅ Modèle AR1 CI chargé


# Filtrer

In [96]:
def filter_df_by_start_date(df: pd.DataFrame, start_date: str = "1990-01-01") -> pd.DataFrame:
    """
    Filtre un DataFrame standardisé (date, true, pred, method)
    en ne gardant que les observations à partir de `start_date`.
    Retourne un DataFrame trié + index propre.
    """
    out = df.copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce")

    out = (
        out[out["date"] >= pd.Timestamp(start_date)]
        .sort_values(["date", "method"])
        .reset_index(drop=True)
    )

    print(f"\n✅ Filtrage appliqué — période: {out['date'].min().date()} → {out['date'].max().date()} | n={len(out)}")
    print("Méthodes présentes :", sorted(out["method"].unique().tolist()))
    print("\nAperçu post-filtrage :")
    print(out.head(10))

    return out

In [97]:
df_pred_long_1990 = filter_df_by_start_date(
    df_b,
    start_date="1990-01-01"
)


✅ Filtrage appliqué — période: 1990-01-01 → 2025-08-01 | n=5070
Méthodes présentes : ['AR1']

Aperçu post-filtrage :
        date  true      pred        lo        hi method
0 1990-01-01   0.0 -0.176432 -1.256478  0.232328    AR1
1 1990-01-01   0.0 -0.323134 -1.310293  0.030914    AR1
2 1990-01-01   0.0 -0.480453 -1.549056 -0.120758    AR1
3 1990-01-01   0.0 -0.129681 -1.092598  0.339040    AR1
4 1990-01-01   0.0 -0.288392 -1.258280  0.086019    AR1
5 1990-01-01   0.0 -0.065229 -0.928914  0.340014    AR1
6 1990-01-01   0.0 -0.150857 -0.995133  0.268030    AR1
7 1990-01-01   0.0 -0.326491 -1.089284 -0.057201    AR1
8 1990-01-01   0.0 -0.079296 -0.647683  0.327721    AR1
9 1990-01-01   0.0 -0.084223 -0.632761  0.378900    AR1


# Analyser la performance prédictive des modèles

## Comparaison des données de test et de prévision

In [103]:
import pandas as pd
from utilsforecast.plotting import plot_series

# =========================
# 0) Base: df_pred_long_1990 (date,true,pred,lo,hi,method)
# =========================
df_std = df_pred_long_1990.copy()
df_std["date"] = pd.to_datetime(df_std["date"], errors="coerce")

# =========================
# 1) DEDOUBLONNER : 1 ligne par (date, method)
#    -> médiane sur pred/lo/hi, true prend la 1ère (identique normalement)
# =========================
df_dedup = (
    df_std
    .groupby(["date", "method"], as_index=False)
    .agg(
        true=("true", "first"),
        pred=("pred", "median"),
        lo=("lo", "median"),
        hi=("hi", "median"),
    )
    .sort_values(["date", "method"])
    .reset_index(drop=True)
)

# Check
dup = df_dedup.duplicated(subset=["date", "method"]).sum()
print("Doublons après agg =", dup)

# =========================
# 2) Passer en WIDE (une ligne par date)
# =========================
df_true = (
    df_dedup[["date", "true"]]
    .drop_duplicates("date")
    .rename(columns={"true": "y_obs"})
)

pred_w = df_dedup.pivot(index="date", columns="method", values="pred").reset_index()
lo_w   = df_dedup.pivot(index="date", columns="method", values="lo").reset_index()
hi_w   = df_dedup.pivot(index="date", columns="method", values="hi").reset_index()

# Renommer colonnes (méthode -> y_hat_xxx / lo / hi)
for m in [c for c in pred_w.columns if c != "date"]:
    pred_w = pred_w.rename(columns={m: f"y_hat_{m}"})
for m in [c for c in lo_w.columns if c != "date"]:
    lo_w = lo_w.rename(columns={m: f"y_hat_{m}_lo_95"})
for m in [c for c in hi_w.columns if c != "date"]:
    hi_w = hi_w.rename(columns={m: f"y_hat_{m}_hi_95"})

df_ar_forecasts = (
    df_true
    .merge(pred_w, on="date", how="inner")
    .merge(lo_w,   on="date", how="left")
    .merge(hi_w,   on="date", how="left")
    .assign(series_id="UNRATE")
    .sort_values("date")
    .reset_index(drop=True)
)

# =========================
# 3) df_obs + df_fcst (format utilsforecast)
# =========================
df_obs = (
    df_ar_forecasts
    .rename(columns={"series_id": "unique_id", "date": "ds", "y_obs": "y"})
    [["unique_id", "ds", "y"]]
)

methods = sorted(df_dedup["method"].unique().tolist())  # ex: ["AR1"]
rename_map = {"series_id": "unique_id", "date": "ds"}

fcst_cols = ["unique_id", "ds"]
for m in methods:
    rename_map[f"y_hat_{m}"] = m
    rename_map[f"y_hat_{m}_lo_95"] = f"{m}-lo-95"
    rename_map[f"y_hat_{m}_hi_95"] = f"{m}-hi-95"
    fcst_cols += [m, f"{m}-lo-95", f"{m}-hi-95"]

df_fcst = (
    df_ar_forecasts
    .rename(columns=rename_map)
    [fcst_cols]
)

# =========================
# 4) Plot + renommage légende
# =========================
fig = plot_series(
    df=df_obs,
    forecasts_df=df_fcst,
    level=[95],
    engine="plotly",
).update_layout(height=400)

for trace in fig.data:
    n = (trace.name or "")
    nl = n.lower()

    if n == "y":
        trace.name = "Unemployment rate (%)"
    elif n in methods:
        trace.name = "AutoRegressive AR(1)" if n.upper() == "AR1" else f"Forecast {n}"
    elif "level_95" in nl:
        trace.name = "95% Prediction Interval (median-aggregated)"

fig.show()

Doublons après agg = 0
